# Luckyscent webscrapen
Scrapes naam, merk, notes, stijl, beschrijving, prijs, concentratie, gender, land.

In [1]:
# Installeer benodigde packages
!pip install requests beautifulsoup4 pandas

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Laptop\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
!pip install beautifulsoup4


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Laptop\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import subprocess
subprocess.run([r"C:\Users\Laptop\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe", 
                "-m", "pip", "install", "beautifulsoup4"], capture_output=True)

CompletedProcess(args=['C:\\Users\\Laptop\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\python.exe', '-m', 'pip', 'install', 'beautifulsoup4'], returncode=0, stdout=b'Defaulting to user installation because normal site-packages is not writeable\r\nCollecting beautifulsoup4\r\n  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)\r\nCollecting soupsieve>=1.6.1 (from beautifulsoup4)\r\n  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)\r\nRequirement already satisfied: typing-extensions>=4.0.0 in c:\\users\\laptop\\appdata\\local\\packages\\pythonsoftwarefoundation.python.3.13_qbz5n2kfra8p0\\localcache\\local-packages\\python313\\site-packages (from beautifulsoup4) (4.12.2)\r\nUsing cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)\r\nUsing cached soupsieve-2.8.3-py3-none-any.whl (37 kB)\r\nInstalling collected packages: soupsieve, beautifulsoup4\r\n\r   -------------------- ------------------- 1/2 [beau

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
BASE_URL = "https://www.luckyscent.com"
DELAY = 1.5  # seconden tussen requests

print('Imports OK')

Imports OK


## Verzamel alle product-URLs

In [13]:
def get_product_urls(max_pages=None):
    """Pagineert door /fragrances en verzamelt alle product-URLs."""
    urls = []
    cursor_url = f"{BASE_URL}/fragrances"
    page = 0

    while cursor_url:
        page += 1
        if max_pages and page > max_pages:
            break

        print(f"Listing pagina {page}: {cursor_url}")
        resp = requests.get(cursor_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")

        for a in soup.select("a[href^='/products/']"):
            full = BASE_URL + a["href"]
            if full not in urls:
                urls.append(full)

        next_link = soup.select_one("a[href*='cursor=']") 
        cursor_url = BASE_URL + next_link["href"] if next_link else None

        time.sleep(DELAY)

    print(f"\nTotaal gevonden: {len(urls)} product-URLs")
    return urls

In [14]:
# MAX_PAGES = 3  # voor een snelle test
MAX_PAGES = None  # alle pagina's

product_urls = get_product_urls(max_pages=MAX_PAGES)
print(product_urls[:5])  # preview

Listing pagina 1: https://www.luckyscent.com/fragrances
Listing pagina 2: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDE5NzIxNzgwNDYwOSwibGFzdF92YWx1ZSI6IjIwMjYtMDMtMTggMDA6MzA6MTUuMDAwMDAwIiwib2Zmc2V0IjozNX0%3D
Listing pagina 3: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDE4MDc4MzUzODQ5NywibGFzdF92YWx1ZSI6IjIwMjYtMDItMjggMDE6NTA6MDYuMDAwMDAwIiwib2Zmc2V0Ijo3MX0%3D
Listing pagina 4: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDE3NTkyODY5NzE1MywibGFzdF92YWx1ZSI6IjIwMjYtMDItMjEgMTk6NDA6NTIuMDAwMDAwIiwib2Zmc2V0IjoxMDd9
Listing pagina 5: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDE1OTYyNDkxMzIxNywibGFzdF92YWx1ZSI6IjIwMjYtMDItMDMgMjA6NDA6NTguMDAwMDAwIiwib2Zmc2V0IjoxNDN9
Listing pagina 6: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDEyMjk4MDg4NDgwMSwibGFzdF92YWx1ZSI6IjIwMjYtMDEtMTMgMjI6MTE6MDYuMDAwMDAwIiwib2Zmc2V0IjoxNzl9
Listi

## Scrape elke productpagina

In [15]:
def scrape_product(url):
    """Scraped één productpagina en geeft een dict terug."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Naam
        name_tag = soup.select_one("h1")
        name = name_tag.get_text(strip=True) if name_tag else ""

        # Merk
        brand_tag = soup.select_one("a[href*='/brands/']")
        brand = brand_tag.get_text(strip=True) if brand_tag else ""

        # Prijs
        price = ""
        for tag in soup.find_all(string=True):
            text = tag.strip()
            if text.startswith("$") and len(text) < 10:
                price = text
                break

        # Concentratie, gender, land via anchor links
        concentration, gender, country, released = "", "", "", ""
        for a in soup.select("a[href*='fragrances?']"):
            href = a["href"]
            txt = a.get_text(strip=True)
            if "concentration=" in href:
                concentration = txt
            elif "gender=" in href:
                gender = txt
            elif "country=" in href:
                country = txt
            elif "year_released=" in href:
                released = txt

        # Notes
        notes = []
        for a in soup.select("a[href*='notes=']"):
            txt = a.get_text(strip=True)
            if txt and txt not in notes:
                notes.append(txt)

        # Stijl
        styles = []
        for a in soup.select("a[href*='style=']"):
            txt = a.get_text(strip=True)
            if txt and txt not in styles:
                styles.append(txt)

        # Beschrijving (The Scoop)
        description = ""
        scoop = soup.find(
            lambda tag: tag.name in ["h3", "h4", "h2"] and "Scoop" in tag.get_text()
        )
        if scoop:
            p = scoop.find_next_sibling("p") or scoop.find_next("p")
            if p:
                description = p.get_text(strip=True)

        return {
            "name": name,
            "brand": brand,
            "concentration": concentration,
            "gender": gender,
            "country": country,
            "released": released,
            "price": price,
            "notes": ", ".join(notes),
            "style": ", ".join(styles),
            "description": description,
            "url": url,
        }

    except Exception as e:
        print(f"Fout bij {url}: {e}")
        return None

In [16]:
# Snelle test op 1 product
test = scrape_product("https://www.luckyscent.com/products/kun-amo-by-jeroboam")
for k, v in test.items():
    print(f"{k}: {v}")

name: Kun Amo
brand: Brands
concentration: Extrait
gender: Unisex
country: France
released: 2025
price: $
notes: Mandarin, Bergamot, Almond, Crunchy Pear, Jasmine, Sea Spray, Raspberry, Cedarwood, Sugar, White Musks, Ambery Woods, Cashmeran
style: Floral - Fruity, Fruity, Powdery / Soft, Sweet, Woody - Amber
description: Jeroboam
url: https://www.luckyscent.com/products/kun-amo-by-jeroboam


## Alles scrapen en opslaan

In [17]:
results = []

for i, url in enumerate(product_urls):
    print(f"[{i+1}/{len(product_urls)}] {url}")
    data = scrape_product(url)
    if data:
        results.append(data)
    time.sleep(DELAY)

    # Checkpoint elke 100 producten
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv("luckyscent_checkpoint.csv", index=False)
        print(f"  --> Checkpoint opgeslagen ({len(results)} producten)")

print(f"\nKlaar! {len(results)} producten gescraped.")

[1/3354] https://www.luckyscent.com/products/fragrance-fitting
[2/3354] https://www.luckyscent.com/products/kun-amo-by-jeroboam
[3/3354] https://www.luckyscent.com/products/mango-sticky-rice-by-dannam
[4/3354] https://www.luckyscent.com/products/dandelion-butter-by-clue
[5/3354] https://www.luckyscent.com/products/white-rice-by-dannam
[6/3354] https://www.luckyscent.com/products/pear-pavlova-by-french-cowboy
[7/3354] https://www.luckyscent.com/products/chloe-sevigny-little-flower-by-regime-des-fleurs
[8/3354] https://www.luckyscent.com/products/tears-by-regime-des-fleurs
[9/3354] https://www.luckyscent.com/products/unknown-pleasures-by-kerosene
[10/3354] https://www.luckyscent.com/products/april-theme-april-showers-by-perfume-room-smell-club
[11/3354] https://www.luckyscent.com/products/wild-vetiver-by-creed
[12/3354] https://www.luckyscent.com/products/molecule-01-champaca-by-escentric-molecules
[13/3354] https://www.luckyscent.com/products/dia-x-meg-webster-by-comme-des-garcons
[14/3

## Opslaan als CSV

In [18]:
df = pd.DataFrame(results)
df.to_csv("luckyscent_fragrances.csv", index=False)
print(f"Opgeslagen: luckyscent_fragrances.csv ({len(df)} rijen)")
df.head()

Opgeslagen: luckyscent_fragrances.csv (3353 rijen)


,name,brand,concentration,gender,country,released,price,notes,style,description,url
0,Fragrance Fitting - Custom Sample Pack,Brands,,,,,$,,,Let us help you find your next signature scent...,https://www.luckyscent.com/products/fragrance-...
1,Kun Amo,Brands,Extrait,Unisex,France,2025,$,"Mandarin, Bergamot, Almond, Crunchy Pear, Jasm...","Floral - Fruity, Fruity, Powdery / Soft, Sweet...",Jeroboam,https://www.luckyscent.com/products/kun-amo-by...
2,Mango Sticky Rice,Brands,Eau de Parfum,Unisex,,2026,$,"Ripe Mango, Sticky Rice, Coconut Milk","Creamy, Fruity, Gourmand, Milky, Sweet, Tropic...",French Cowboy,https://www.luckyscent.com/products/mango-stic...
3,Dandelion Butter,Brands,,Unisex,United States,2025,$?,"Pollen, Dandelion Greens, Yellow Dandelion, Sn...","Floral, Floral - Light, Green / Herbaceous, Mi...",The boundlessly creative noses behind Clue dra...,https://www.luckyscent.com/products/dandelion-...
4,White Rice,Brands,Eau de Parfum,Unisex,Vietnam,2023,$,"Rice, Pandan, Orris, Jasmine, White Musk, Ceda...","Creamy, Floral - Light, Gourmand, Milky, Powde...",d'Annam,https://www.luckyscent.com/products/white-rice...


In [19]:
# Snelle statistieken
print(f"Totaal producten: {len(df)}")
print(f"Unieke merken: {df['brand'].nunique()}")
print(f"Missende notes: {df['notes'].eq('').sum()}")
print(f"Missende beschrijving: {df['description'].eq('').sum()}")
print("\nTop 10 merken:")
print(df['brand'].value_counts().head(10))

Totaal producten: 3353
Unieke merken: 2
Missende notes: 350
Missende beschrijving: 6

Top 10 merken:
brand
Brands    3352
             1
Name: count, dtype: int64
